# Attack Family vs Baseline Evaluation
This notebook analyzes poisoned evaluation results and compares attack families against a clean baseline for key retrieval metrics.

## 1. Import Required Libraries
Import libraries for data manipulation, plotting, and JSON loading.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=1.05)

## 2. Load Evaluation Data
Load the JSON files for each attack family and baseline, then normalize records into a combined DataFrame.

In [ ]:
root = Path(".")
evaluation_files = {
    "Text-only (with metadata) poisoning": root / "evaluation_results_poisoned_text_only.json",
    "Image-only poisoning": root / "evaluation_results_poisoned_image_only.json",
    "Hybrid image-text poisoning": root / "evaluation_results_poisoned_hybrid.json",
    "Baseline (correct cases)": root / "evaluation_results_baseline.json",
}

loaded_frames = []
missing_files = []

for label, path in evaluation_files.items():
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, dict):
            if "results" in payload:
                records = payload["results"]
            elif "records" in payload:
                records = payload["records"]
            else:
                raise ValueError(f"Unrecognized JSON schema in {path}")
        elif isinstance(payload, list):
            records = payload
        else:
            raise ValueError(f"Unsupported JSON root type in {path}")

        df = pd.json_normalize(records)
        df["attack_family"] = label
        loaded_frames.append(df)
    else:
        missing_files.append(path)

if missing_files:
    print("Missing evaluation files. Update the paths or add files for:")
    for missing in missing_files:
        print(f" - {missing}")

if loaded_frames:
    results_df = pd.concat(loaded_frames, ignore_index=True)
else:
    raise FileNotFoundError("No evaluation files were loaded. Check the paths above.")

results_df.head()

## 3. Preprocess Data by Attack Family
Convert metrics to numeric and compute aggregated mean values per attack family and baseline.

In [ ]:
results_df["ndcg"] = pd.to_numeric(results_df["ndcg"], errors="coerce")
results_df["recall"] = pd.to_numeric(results_df["recall"], errors="coerce")
results_df["mrr"] = pd.to_numeric(results_df["mrr"], errors="coerce")

aggregated = (
    results_df.groupby("attack_family")[ ["ndcg", "recall", "mrr"] ]
    .mean()
    .rename(columns={"ndcg": "avg_ndcg", "recall": "avg_recall", "mrr": "avg_mrr"})
    .reset_index()
)

aggregated

## 4. Generate Individual Plots per Attack Family
Plot each poisoning attack family against the baseline for NDCG, recall, and MRR.

In [ ]:
non_baseline = aggregated[aggregated["attack_family"] != "Baseline"]
baseline = aggregated[aggregated["attack_family"] == "Baseline"]
metric_names = {"avg_ndcg": "Average NDCG", "avg_recall": "Average Recall", "avg_mrr": "Average MRR"}

for label in non_baseline["attack_family"]:
    comparison = aggregated[aggregated["attack_family"].isin([label, "Baseline"])]
    if comparison.shape[0] < 2:
        continue

    melted = comparison.melt(
        id_vars=["attack_family"],
        value_vars=["avg_ndcg", "avg_recall", "avg_mrr"],
        var_name="metric",
        value_name="value",
    )
    melted["metric"] = melted["metric"].map(metric_names)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(
        data=melted,
        x="metric",
        y="value",
        hue="attack_family",
        palette=["#377eb8", "#e41a1c"],
        ax=ax,
    )
    ax.set_title(f"{label} vs Baseline")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Metric value")
    ax.set_xlabel("")
    ax.legend(title="Dataset", loc="upper right")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", label_type="edge", padding=2)
    fig.tight_layout()
    plt.show()

## 5. Create Combination Plot
Generate a single combined chart that compares all attack families and baseline across the three metrics.

In [ ]:
plot_df = aggregated.melt(
    id_vars=["attack_family"],
    value_vars=["avg_ndcg", "avg_recall", "avg_mrr"],
    var_name="metric",
    value_name="value",
)
plot_df["metric"] = plot_df["metric"].map(metric_names)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x="attack_family",
    y="value",
    hue="metric",
    palette="Set2",
    alpha=0.95,
    edgecolor="black",
    ax=ax,
)

ax.set_title("Attack families vs Baseline: aggregated retrieval metrics")
ax.set_xlabel("Attack family")
ax.set_ylabel("Average metric value")
ax.set_ylim(0, 1)
ax.legend(title="Metric", loc="upper right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", label_type="edge", padding=2)
fig.tight_layout()
plt.show()

## 6. Build Summation Table
Create a summary table showing average metric values and deltas relative to baseline.

In [ ]:
summary = aggregated.copy()

if not baseline.empty:
    baseline_row = baseline.iloc[0]
    for metric_key in ["avg_ndcg", "avg_recall", "avg_mrr"]:
        summary[f"delta_vs_baseline_{metric_key}"] = summary[metric_key] - baseline_row[metric_key]

summary_style = (
    summary.style
    .format(
        {
            "avg_ndcg": "{:.3f}",
            "avg_recall": "{:.3f}",
            "avg_mrr": "{:.3f}",
            "delta_vs_baseline_avg_ndcg": "{:+.3f}",
            "delta_vs_baseline_avg_recall": "{:+.3f}",
            "delta_vs_baseline_avg_mrr": "{:+.3f}",
        }
    )
    .background_gradient(subset=["avg_ndcg", "avg_recall", "avg_mrr"], cmap="Blues")
    .background_gradient(
        subset=["delta_vs_baseline_avg_ndcg", "delta_vs_baseline_avg_recall", "delta_vs_baseline_avg_mrr"],
        cmap="RdYlGn",
    )
)

summary_style